In [ ]:
!pip install -q onnxruntime tokenizers huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 54.5 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "onnx-models/all-MiniLM-L6-v2-onnx"

onnx_model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename="model.onnx"
)

tokenizer_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename="tokenizer.json"
)

print("✅ ONNX model downloaded")
print("✅ Tokenizer downloaded")
print("Model:", onnx_model_path)
print("Tokenizer:", tokenizer_path)

model.onnx: reconstructing file:   0%|          |  0.00B / 90.4MB            

model.onnx: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

✅ ONNX model downloaded
✅ Tokenizer downloaded
Model: /root/.cache/huggingface/hub/models--onnx-models--all-MiniLM-L6-v2-onnx/snapshots/75251058ddd779e3a744f87fdf63fb39681aec16/model.onnx
Tokenizer: /root/.cache/huggingface/hub/models--onnx-models--all-MiniLM-L6-v2-onnx/snapshots/75251058ddd779e3a744f87fdf63fb39681aec16/tokenizer.json


In [ ]:
# import onnxruntime as ort

# session = ort.InferenceSession(
#     onnx_model_path,
#     providers=["CPUExecutionProvider"]
# )

# print("Inputs:")
# for x in session.get_inputs():
#     print(
#         x.name,
#         x.shape,
#         x.type
#     )

# print("\nOutputs:")
# for x in session.get_outputs():
#     print(
#         x.name,
#         x.shape,
#         x.type
#     )

In [ ]:
import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer


class ONNXEmbeddingModel:

    def __init__(
        self,
        model_path,
        tokenizer_path,
        max_length=256
    ):

        self.tokenizer = Tokenizer.from_file(
            tokenizer_path
        )

        self.tokenizer.enable_truncation(
            max_length=max_length
        )

        self.tokenizer.enable_padding()

        self.session = ort.InferenceSession(
            model_path,
            providers=["CPUExecutionProvider"]
        )

    def encode(
        self,
        sentences,
        batch_size=32,
        show_progress_bar=False,
        normalize_embeddings=True
    ):

        if isinstance(sentences, str):
            sentences = [sentences]

        all_embeddings = []

        for start in range(
            0,
            len(sentences),
            batch_size
        ):

            batch = sentences[
                start:start + batch_size
            ]

            encodings = self.tokenizer.encode_batch(
                batch
            )

            input_ids = np.asarray(
                [e.ids for e in encodings],
                dtype=np.int64
            )

            attention_mask = np.asarray(
                [e.attention_mask for e in encodings],
                dtype=np.int64
            )

            token_type_ids = np.asarray(
                [e.type_ids for e in encodings],
                dtype=np.int64
            )

            outputs = self.session.run(
                ["sentence_embedding"],
                {
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                    "token_type_ids": token_type_ids
                }
            )

            embeddings = outputs[0].astype(
                np.float32
            )

            if normalize_embeddings:

                norms = np.linalg.norm(
                    embeddings,
                    axis=1,
                    keepdims=True
                )

                embeddings = embeddings / np.clip(
                    norms,
                    1e-12,
                    None
                )

            all_embeddings.append(
                embeddings
            )

        return np.vstack(
            all_embeddings
        )

    def get_sentence_embedding_dimension(self):
        return 384

In [ ]:
# onnx_embedding_model = ONNXEmbeddingModel(
#     onnx_model_path,
#     tokenizer_path
# )

# print("✅ ONNX embedding model loaded!")
# print(
#     "Embedding dimension:",
#     onnx_embedding_model.get_sentence_embedding_dimension()
# )

In [ ]:
!pip install -q pandas numpy scikit-learn
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "bitext/Bitext-retail-ecommerce-llm-chatbot-training-dataset"
)

print(dataset)

README.md:   0%|          | 0.00/10.1k [00:00<?, ?B/s]

bitext-retail-ecommerce-llm-chatbot-trai(…): reconstructing file:   0%|          |  0.00B / 42.6MB            

bitext-retail-ecommerce-llm-chatbot-trai(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/44884 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'intent', 'category', 'tags', 'response'],
        num_rows: 44884
    })
})


In [ ]:
df = dataset["train"].to_pandas()

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head(15))

Shape: (44884, 5)
Columns:
['instruction', 'intent', 'category', 'tags', 'response']


,instruction,intent,category,tags,response
0,I got to add an item to the cart,add_product,CART,BL,I'll get right on it! I'm here to assist you i...
1,wanna add fucking products to the basket can h...,add_product,CART,BCIMQWZ,I sincerely apologize if you've encountered an...
2,i have to add products to the basket i ned help,add_product,CART,BCMQZ,You bet! I'm here to assist you in adding prod...
3,di like to add products to the cart could i ge...,add_product,CART,BCILMPQZ,Indeed! I'm here to assist you in adding produ...
4,"I need to add an item to the cart , where do I...",add_product,CART,BCILZ,I'll take care of it! I'm here to help you wit...
5,help me add products to the basket,add_product,CART,BM,I'm happy to help! I'm here to guide you throu...
6,"I need to add a fucking product to the basket,...",add_product,CART,BCIWZ,"I understand the inconvenience, and I'm sorry ..."
7,"I have to add items to the basket, where could...",add_product,CART,BCILMP,Indeed! I'm here to help you with adding items...
8,I would like to add a fucking item to the bask...,add_product,CART,BCILPWZ,I genuinely apologize for any frustration caus...
9,i got to add a fucking item ot the cart can i ...,add_product,CART,BCILQWZ,I sincerely apologize for any frustration you ...


In [ ]:
for col in df.columns:
    print("\nCOLUMN:", col)
    print(df[col].dropna().astype(str).iloc[0][:300])


COLUMN: instruction
I got to add an item to the cart

COLUMN: intent
add_product

COLUMN: category
CART

COLUMN: tags
BL

COLUMN: response
I'll get right on it! I'm here to assist you in adding an item to your cart. Here's a step-by-step guide to help you out:

1. Browse through our {{Product Catalog}} to find the item you want to add to your cart.
2. Once you've found the item, click on the "{{Add to Cart}}" button or select the desir


In [ ]:
# 1. Basic information
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

# 2. Number of unique intents and categories
print("\nNumber of unique intents:",
      df["intent"].nunique())

print("Number of unique categories:",
      df["category"].nunique())

# 3. List all intents
print("\nINTENTS:")
print(sorted(df["intent"].unique().tolist()))

# 4. List all categories
print("\nCATEGORIES:")
print(sorted(df["category"].unique().tolist()))

# 5. Missing values
print("\nMISSING VALUES:")
print(df.isnull().sum())

# 6. Duplicate rows
print("\nDuplicate rows:",
      df.duplicated().sum())

# 7. Intent distribution
print("\nINTENT DISTRIBUTION:")
print(
    df["intent"]
    .value_counts()
    .sort_values(ascending=False)
)

# 8. Category distribution
print("\nCATEGORY DISTRIBUTION:")
print(
    df["category"]
    .value_counts()
    .sort_values(ascending=False)
)

Shape: (44884, 5)

Columns:
['instruction', 'intent', 'category', 'tags', 'response']

Number of unique intents: 46
Number of unique categories: 13

INTENTS:
['add_product', 'availability', 'availability_in_store', 'availability_online', 'cancel_order', 'change_account', 'change_order', 'close_account', 'customer_service', 'damaged_delivery', 'delivery_issue', 'delivery_time', 'exchange_product', 'exchange_product_in_store', 'human_agent', 'missing_item', 'open_account', 'order_history', 'pay', 'payment_issue', 'payment_methods', 'product_information', 'product_issue', 'recover_password', 'refund_policy', 'refund_status', 'remove_product', 'request_invoice', 'request_refund', 'request_right_to_rectification', 'return_policy', 'return_product', 'return_product_in_store', 'return_product_online', 'sales_period', 'shipping_costs', 'store_location', 'store_opening_hours', 'submit_feedback', 'submit_product_feedback', 'submit_product_idea', 'technical_issue', 'track_delivery', 'track_order'

In [ ]:
# Show 5 examples for every intent

examples = (
    df.groupby("intent", group_keys=False)
      .head(5)
      [["instruction", "intent", "category"]]
)

display(examples)

,instruction,intent,category
0,I got to add an item to the cart,add_product,CART
1,wanna add fucking products to the basket can h...,add_product,CART
2,i have to add products to the basket i ned help,add_product,CART
3,di like to add products to the cart could i ge...,add_product,CART
4,"I need to add an item to the cart , where do I...",add_product,CART
...,...,...,...
43888,"I got a fucking product in the wrong color , c...",wrong_item,DELIVERY
43889,some items in my delivery are incorrect can uh...,wrong_item,DELIVERY
43890,fucking products in my delivery are wrong i ne...,wrong_item,DELIVERY
43891,"the fucking items I ordered are incorrect, can...",wrong_item,DELIVERY


In [ ]:
print("Rows:", len(df))
print("Intents:", df["intent"].nunique())
print("Categories:", df["category"].nunique())

print("\nIntent examples:")
display(
    df[["instruction", "intent", "category", "response"]].sample(
        10,
        random_state=42
    )
)

Rows: 44884
Intents: 46
Categories: 13

Intent examples:


,instruction,intent,category,response
33698,"I'd like information about the fucking sales, ...",sales_period,SALES,I genuinely apologize for the inappropriate la...
44212,one of the products in mh package is incorrect...,wrong_item,DELIVERY,I'm sorry to hear that you received an incorre...
14749,im missing some fucking products i ordered how...,missing_item,DELIVERY,"We regret the inconvenience, and we're sorry f..."
36515,im looking for information aboyt ur fucking sh...,store_opening_hours,STORE,I genuinely apologize for any frustration caus...
24333,I'd like information about the fucking status ...,refund_status,RETURNS,I sincerely apologize for the inconvenience an...
29325,I'd like information about your fucking return...,return_policy,RETURNS,We recognize the inconvenience and apologize f...
5770,i boughty a product but i chose a wrong model ...,change_order,ORDER,Our apologies for any inconvenience caused by ...
16295,I'd like to review my purchase history how cou...,order_history,ACCOUNT,Definitely! I can help you with that. Reviewin...
26125,i have to download a fucking invoice could i g...,request_invoice,ORDER,I deeply apologize for any frustration or inco...
7707,can ya help me calling fucking customer support,customer_service,CONTACT,I am truly sorry to hear about the frustration...


In [ ]:
print(
    df.groupby("intent")["instruction"]
      .count()
      .sort_values()
)

intent
missing_item                       721
availability_in_store              756
delivery_time                      920
store_location                     924
request_refund                     957
add_product                        957
change_order                       961
availability                       972
shipping_costs                     974
product_information                987
open_account                       987
change_account                     987
order_history                      988
track_order                        988
exchange_product                   988
request_right_to_rectification     989
pay                                990
submit_feedback                    991
exchange_product_in_store          991
store_opening_hours                992
product_issue                      992
damaged_delivery                   992
customer_service                   992
remove_product                     993
availability_online                993
recover_password  

In [ ]:
from sklearn.model_selection import train_test_split

X = df["instruction"]
y = df["intent"]

# 80% train, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Split remaining 20% into 10% validation and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 35907
Validation: 4488
Testing: 4489


In [ ]:
print("Train intents:", y_train.nunique())
print("Validation intents:", y_val.nunique())
print("Test intents:", y_test.nunique())

Train intents: 46
Validation intents: 46
Test intents: 46


In [ ]:
print("\nExample training rows:")
display(
    pd.DataFrame({
        "instruction": X_train.head(10).values,
        "intent": y_train.head(10).values
    })
)


Example training rows:


,instruction,intent
0,"I have to check your fucking return policy, ho...",return_policy
1,how to report fucking delivery problems with m...,delivery_issue
2,"I'd like to pay with a fucking debit card, wh...",payment_methods
3,idont need my fucking profile i wanna delete it,close_account
4,can you help me to review my order history?,order_history
5,i cant make payments i need assistance repofti...,payment_issue
6,I'd like to send fucking feedback about your c...,submit_feedback
7,"I need to seeitem availability, could you help...",availability
8,the app doesnt work on my cell phone could uhe...,technical_issue
9,need to see ur fucking refund policy,refund_policy


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Create TF-IDF features
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)


Train TF-IDF shape: (35907, 9048)
Validation TF-IDF shape: (4488, 9048)
Test TF-IDF shape: (4489, 9048)


In [ ]:
classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

classifier.fit(
    X_train_tfidf,
    y_train
)

print("Baseline classifier trained successfully!")

Baseline classifier trained successfully!


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# Validation predictions
val_predictions = classifier.predict(X_val_tfidf)

val_accuracy = accuracy_score(
    y_val,
    val_predictions
)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_val,
    val_predictions,
    average="macro",
    zero_division=0
)

print("Validation Accuracy :", round(val_accuracy, 4))
print("Validation Precision:", round(precision, 4))
print("Validation Recall   :", round(recall, 4))
print("Validation Macro-F1  :", round(f1, 4))

Validation Accuracy : 0.9873
Validation Precision: 0.9876
Validation Recall   : 0.9874
Validation Macro-F1  : 0.9874


In [ ]:
# Final evaluation on the untouched test set

test_predictions = classifier.predict(X_test_tfidf)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

test_precision, test_recall, test_f1, _ = (
    precision_recall_fscore_support(
        y_test,
        test_predictions,
        average="macro",
        zero_division=0
    )
)

print("TEST RESULTS")
print("------------------------------")
print("Accuracy :", round(test_accuracy, 4))
print("Precision:", round(test_precision, 4))
print("Recall   :", round(test_recall, 4))
print("Macro-F1 :", round(test_f1, 4))

TEST RESULTS
------------------------------
Accuracy : 0.9851
Precision: 0.9858
Recall   : 0.9853
Macro-F1 : 0.9852


In [ ]:
print(
    classification_report(
        y_test,
        test_predictions,
        zero_division=0
    )
)

                                precision    recall  f1-score   support

                   add_product       1.00      1.00      1.00        95
                  availability       0.98      0.99      0.98        97
         availability_in_store       0.99      0.99      0.99        75
           availability_online       0.99      1.00      1.00       100
                  cancel_order       1.00      0.98      0.99       100
                change_account       0.97      1.00      0.98        98
                  change_order       0.99      1.00      0.99        96
                 close_account       1.00      0.97      0.98       100
              customer_service       1.00      0.99      0.99        99
              damaged_delivery       0.99      0.96      0.97        99
                delivery_issue       1.00      0.99      0.99       100
                 delivery_time       1.00      1.00      1.00        92
              exchange_product       1.00      1.00      1.00  

In [ ]:
embedding_model = ONNXEmbeddingModel(
    onnx_model_path,
    tokenizer_path
)

print("✅ ONNX embedding model loaded!")
print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

✅ ONNX embedding model loaded!
Embedding dimension: 384


In [ ]:
X_train_embeddings = embedding_model.encode(
    X_train.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", X_train_embeddings.shape)

Embedding shape: (35907, 384)


In [ ]:
from sklearn.linear_model import LogisticRegression

embedding_classifier = LogisticRegression(
    max_iter=1000,
    random_state=42
)

embedding_classifier.fit(
    X_train_embeddings,
    y_train
)

print("Embedding classifier trained successfully!")

Embedding classifier trained successfully!


In [ ]:
X_val_embeddings = embedding_model.encode(
    X_val.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Validation embedding shape:",
    X_val_embeddings.shape
)

Validation embedding shape: (4488, 384)


In [ ]:
embedding_val_predictions = embedding_classifier.predict(
    X_val_embeddings
)

embedding_val_accuracy = accuracy_score(
    y_val,
    embedding_val_predictions
)

embedding_val_precision, embedding_val_recall, embedding_val_f1, _ = (
    precision_recall_fscore_support(
        y_val,
        embedding_val_predictions,
        average="macro",
        zero_division=0
    )
)

print("EMBEDDING MODEL — VALIDATION")
print("--------------------------------")
print("Accuracy :", round(embedding_val_accuracy, 4))
print("Precision:", round(embedding_val_precision, 4))
print("Recall   :", round(embedding_val_recall, 4))
print("Macro-F1 :", round(embedding_val_f1, 4))

EMBEDDING MODEL — VALIDATION
--------------------------------
Accuracy : 0.9898
Precision: 0.99
Recall   : 0.9895
Macro-F1 : 0.9896


In [ ]:
# Create test embeddings
X_test_embeddings = embedding_model.encode(
    X_test.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Test embedding shape:",
    X_test_embeddings.shape
)

Test embedding shape: (4489, 384)


In [ ]:
# Final evaluation of embedding classifier
embedding_test_predictions = embedding_classifier.predict(
    X_test_embeddings
)

embedding_test_accuracy = accuracy_score(
    y_test,
    embedding_test_predictions
)

embedding_test_precision, embedding_test_recall, embedding_test_f1, _ = (
    precision_recall_fscore_support(
        y_test,
        embedding_test_predictions,
        average="macro",
        zero_division=0
    )
)

print("EMBEDDING MODEL — TEST")
print("--------------------------------")
print("Accuracy :", round(embedding_test_accuracy, 4))
print("Precision:", round(embedding_test_precision, 4))
print("Recall   :", round(embedding_test_recall, 4))
print("Macro-F1 :", round(embedding_test_f1, 4))

EMBEDDING MODEL — TEST
--------------------------------
Accuracy : 0.9866
Precision: 0.9871
Recall   : 0.9869
Macro-F1 : 0.9868


In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

cm_embedding = confusion_matrix(
    y_test,
    embedding_test_predictions,
    labels=embedding_classifier.classes_
)

embedding_errors = []

for i, true_label in enumerate(
    embedding_classifier.classes_
):
    for j, predicted_label in enumerate(
        embedding_classifier.classes_
    ):

        if i != j and cm_embedding[i, j] > 0:

            embedding_errors.append({
                "true_intent": true_label,
                "predicted_intent": predicted_label,
                "errors": int(cm_embedding[i, j])
            })

embedding_errors_df = (
    pd.DataFrame(embedding_errors)
      .sort_values("errors", ascending=False)
)

display(embedding_errors_df.head(20))

,true_intent,predicted_intent,errors
35,track_order,track_delivery,7
8,damaged_delivery,wrong_item,5
24,return_policy,return_product_online,4
14,refund_policy,return_policy,3
2,cancel_order,return_product_online,2
5,change_account,open_account,2
26,return_product,return_policy,2
19,request_refund,pay,2
33,track_order,delivery_time,2
17,request_invoice,pay,2


In [ ]:
# retrieval_df = df[
#     ["instruction", "intent", "category", "response"]
# ].copy()

# retrieval_df = retrieval_df.reset_index(drop=True)

# print("Retrieval rows:", len(retrieval_df))
# display(retrieval_df.head())

In [ ]:
# retrieval_embeddings = embedding_model.encode(
#     retrieval_df["instruction"].tolist(),
#     batch_size=64,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )

# print("Retrieval embedding shape:")
# print(retrieval_embeddings.shape)

In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np

# def semantic_search(query, top_k=5):

#     query_embedding = embedding_model.encode(
#         [query],
#         normalize_embeddings=True
#     )

#     similarities = cosine_similarity(
#         query_embedding,
#         retrieval_embeddings
#     )[0]

#     top_indices = np.argsort(
#         similarities
#     )[::-1][:top_k]

#     results = retrieval_df.iloc[
#         top_indices
#     ].copy()

#     results["similarity"] = similarities[
#         top_indices
#     ]

#     return results[
#         [
#             "instruction",
#             "intent",
#             "category",
#             "response",
#             "similarity"
#         ]
#     ].reset_index(drop=True)

In [ ]:
# queries = [
#     "Where is my order?",
#     "My package has not arrived yet",
#     "I want to return something",
#     "My card payment was declined",
#     "Where is my refund?",
#     "I received the wrong item"
# ]

# for query in queries:

#     print("\n" + "=" * 80)
#     print("QUERY:", query)

#     results = semantic_search(
#         query,
#         top_k=3
#     )

#     display(results)

In [ ]:
# Build retrieval corpus using TRAINING data only

retrieval_train_df = df.loc[
    X_train.index,
    ["instruction", "intent", "category", "response"]
].copy().reset_index(drop=True)

print("Retrieval corpus size:", len(retrieval_train_df))
display(retrieval_train_df.head())

Retrieval corpus size: 35907


,instruction,intent,category,response
0,"I have to check your fucking return policy, ho...",return_policy,RETURNS,I genuinely apologize for any frustration or i...
1,how to report fucking delivery problems with m...,delivery_issue,DELIVERY,"It's our responsibility, and we're sorry for a..."
2,"I'd like to pay with a fucking debit card, wh...",payment_methods,PAYMENT,I sincerely apologize for any previous inconve...
3,idont need my fucking profile i wanna delete it,close_account,ACCOUNT,I sincerely apologize for any frustration or i...
4,can you help me to review my order history?,order_history,ACCOUNT,I'll do my best! I'm here to assist you with r...


In [ ]:
retrieval_train_embeddings = embedding_model.encode(
    retrieval_train_df["instruction"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(
    "Retrieval embedding shape:",
    retrieval_train_embeddings.shape
)

Retrieval embedding shape: (35907, 384)


In [ ]:
def semantic_search(
    query,
    top_k=5
):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        retrieval_train_embeddings
    )[0]

    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = retrieval_train_df.iloc[
        top_indices
    ].copy()

    results["similarity"] = similarities[
        top_indices
    ]

    return results[
        [
            "instruction",
            "intent",
            "category",
            "response",
            "similarity"
        ]
    ].reset_index(drop=True)

In [ ]:
queries = [
    "Where is my order?",
    "My package has not arrived yet",
    "I want to return something",
    "My card payment was declined",
    "Where is my refund?",
    "I received the wrong item"
]

for query in queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)

    results = semantic_search(
        query,
        top_k=3
    )

    display(results)


QUERY: Where is my order?


,instruction,intent,category,response,similarity
0,"I got to see where my order is, where could I ...",track_order,ORDER,I'll do my best! I'd be happy to help you trac...,0.906439
1,"I have to see where my order is , could you he...",track_order,ORDER,I'll do my best! I'd be happy to assist you in...,0.879740
2,"I got to check where my order is, could you he...",track_order,ORDER,I'll take care of it! I'm here to assist you i...,0.874632



QUERY: My package has not arrived yet


,instruction,intent,category,response,similarity
0,i have not received my package could ya help m...,delivery_issue,DELIVERY,I'm genuinely sorry for any inconvenience caus...,0.753894
1,i have not received my package i have to repor...,delivery_issue,DELIVERY,I understand your frustration and apologize fo...,0.748325
2,"I haven't received my package, can you help me...",delivery_issue,DELIVERY,I'm sorry to hear that you haven't received yo...,0.737631



QUERY: I want to return something


...

,instruction,intent,category,response,similarity
0,"my card was declined, can you help me notifing...",payment_issue,PAYMENT,I'm sorry to hear that your card was declined ...,0.903879
1,"my credit card was declined, could you help me...",payment_issue,PAYMENT,I'm sorry to hear that your credit card was de...,0.900309
2,"my card was declined, help me to notify of pay...",payment_issue,PAYMENT,I feel the need to apologize for the inconveni...,0.895796



QUERY: Where is my refund?


,instruction,intent,category,response,similarity
0,where do i get a refund,request_refund,RETURNS,I'm sorry to hear that you're looking for a re...,0.902268
1,where to get a refund,request_refund,RETURNS,I appreciate that you're interested in knowing...,0.886556
2,"I got to see the status of my refund, where ca...",refund_status,RETURNS,I understand how important it is for you to tr...,0.874769



QUERY: I received the wrong item


,instruction,intent,category,response,similarity
0,the fucking item i received is wrong,wrong_item,DELIVERY,I genuinely apologize for the inconvenience ca...,0.834476
1,"the item I received is incorrect, can you help...",wrong_item,DELIVERY,I'm sorry to hear that you received the incorr...,0.774932
2,the fucking item i received is incorrect help ...,wrong_item,DELIVERY,I genuinely apologize for the inconvenience ca...,0.774275


In [ ]:
def retrieval_recall_at_k(
    k=5,
    max_queries=None
):

    # Optionally evaluate a subset for speed
    test_indices = np.arange(len(X_test))

    if max_queries is not None:
        test_indices = test_indices[:max_queries]

    hits = 0
    total = len(test_indices)

    for i in test_indices:

        query = X_test.iloc[i]
        true_intent = y_test.iloc[i]

        query_embedding = embedding_model.encode(
            [query],
            normalize_embeddings=True
        )

        similarities = cosine_similarity(
            query_embedding,
            retrieval_train_embeddings
        )[0]

        top_indices = np.argsort(
            similarities
        )[::-1][:k]

        retrieved_intents = (
            retrieval_train_df
            .iloc[top_indices]["intent"]
            .tolist()
        )

        if true_intent in retrieved_intents:
            hits += 1

    return hits / total

In [ ]:
recall_1 = retrieval_recall_at_k(
    k=1,
    max_queries=None
)

print("Retrieval evaluation on full test set")
print("-------------------------------------")
print("Recall@1:", round(recall_1, 4))

Retrieval evaluation on full test set
-------------------------------------
Recall@1: 0.9844


In [ ]:
# recall_1 = retrieval_recall_at_k(
#     k=1,
#     max_queries=500
# )

# recall_3 = retrieval_recall_at_k(
#     k=3,
#     max_queries=500
# )

# recall_5 = retrieval_recall_at_k(
#     k=5,
#     max_queries=500
# )

# print("Retrieval evaluation on 500 test queries")
# print("----------------------------------------")
# print("Recall@1:", round(recall_1, 4))
# print("Recall@3:", round(recall_3, 4))
# print("Recall@5:", round(recall_5, 4))

In [ ]:
def retrieval_mrr(
    max_queries=None
):
    test_indices = np.arange(len(X_test))

    if max_queries is not None:
        test_indices = test_indices[:max_queries]

    reciprocal_ranks = []

    for i in test_indices:

        query = X_test.iloc[i]
        true_intent = y_test.iloc[i]

        query_embedding = embedding_model.encode(
            [query],
            normalize_embeddings=True
        )

        similarities = cosine_similarity(
            query_embedding,
            retrieval_train_embeddings
        )[0]

        ranked_indices = np.argsort(
            similarities
        )[::-1]

        ranked_intents = (
            retrieval_train_df
            .iloc[ranked_indices]["intent"]
            .tolist()
        )

        # Find first occurrence of correct intent
        rank = None

        for position, intent in enumerate(
            ranked_intents,
            start=1
        ):
            if intent == true_intent:
                rank = position
                break

        if rank is not None:
            reciprocal_ranks.append(
                1 / rank
            )
        else:
            reciprocal_ranks.append(0)

    return np.mean(reciprocal_ranks)

In [ ]:
mrr = retrieval_mrr(
    max_queries=500
)

print("MRR:", round(mrr, 4))

MRR: 0.9973


In [ ]:
unseen_queries = [
    ("Where can I see my shipment?", "track_delivery"),
    ("Can you tell me the status of my order?", "track_order"),
    ("My money hasn't come back yet", "refund_status"),
    ("I was charged but my payment failed", "payment_issue"),
    ("The thing delivered to me is not the one I bought", "wrong_item"),
    ("The product arrived broken", "damaged_delivery"),
    ("How do I send something back?", "return_product"),
    ("Can I return this through the website?", "return_product_online"),
    ("Can I swap this item in a physical shop?", "exchange_product_in_store"),
    ("I want to close my account", "close_account"),
]

In [ ]:
for query, expected_intent in unseen_queries:

    results = semantic_search(
        query,
        top_k=3
    )

    predicted_intent = results.iloc[0]["intent"]

    print("\n" + "=" * 70)
    print("Query   :", query)
    print("Expected:", expected_intent)
    print("Top 1   :", predicted_intent)
    print("Score   :", round(
        results.iloc[0]["similarity"], 4
    ))

    print("\nTop 3 intents:")
    print(
        results[
            ["intent", "similarity"]
        ].to_string(index=False)
    )


Query   : Where can I see my shipment?
Expected: track_delivery
Top 1   : track_delivery
Score   : 0.943

Top 3 intents:
        intent  similarity
track_delivery    0.943038
   track_order    0.907128
track_delivery    0.900857

Query   : Can you tell me the status of my order?
Expected: track_order
Top 1   : track_delivery
Score   : 0.7792

Top 3 intents:
        intent  similarity
track_delivery    0.779194
   track_order    0.722269
   track_order    0.708211

Query   : My money hasn't come back yet
Expected: refund_status
Top 1   : refund_policy
Score   : 0.6568

Top 3 intents:
       intent  similarity
refund_policy    0.656777
refund_policy    0.652071
refund_policy    0.651105

Query   : I was charged but my payment failed
Expected: payment_issue
Top 1   : payment_issue
Score   : 0.8862

Top 3 intents:
       intent  similarity
payment_issue    0.886216
payment_issue    0.837583
payment_issue    0.833485

Query   : The thing delivered to me is not the one I bought
Expected: wr

In [ ]:
def get_answer(query, top_k=3):
    results = semantic_search(
        query,
        top_k=top_k
    )

    best_result = results.iloc[0]

    return {
        "answer": best_result["response"],
        "intent": best_result["intent"],
        "category": best_result["category"],
        "similarity": float(best_result["similarity"])
    }

In [ ]:
query = "Where is my order?"

result = get_answer(query)

print("Customer:", query)
print("Answer:", result["answer"])

Customer: Where is my order?
Answer: I'll do my best! I'd be happy to help you track your order. To check the status of your order, you can follow these steps:

1. Visit our website at {{Company Website URL}}.
2. Log in to your account using your registered email address and password.
3. Navigate to the "{{Order History}}" or "{{My Orders}}" section.
4. Look for the specific order you want to track and click on it.
5. You should be able to see the current status of your order, such as "Processing," "Shipped," or "Delivered," along with any tracking information available.

If you're unable to find the information you need or have any other questions, feel free to reach out to our customer support team at {{Customer Support Phone Number}} or via live chat on our website. We're here to assist you every step of the way!


In [ ]:
test_queries = [
    "Where is my order?",
    "My card payment was declined",
    "I want to return an item",
    "Where is my refund?",
    "I received the wrong product",
    "Can I exchange an item in store?"
]

for query in test_queries:

    result = get_answer(query)

    print("\n" + "=" * 80)
    print("Customer:", query)
    print("Answer:", result["answer"])


Customer: Where is my order?
Answer: I'll do my best! I'd be happy to help you track your order. To check the status of your order, you can follow these steps:

1. Visit our website at {{Company Website URL}}.
2. Log in to your account using your registered email address and password.
3. Navigate to the "{{Order History}}" or "{{My Orders}}" section.
4. Look for the specific order you want to track and click on it.
5. You should be able to see the current status of your order, such as "Processing," "Shipped," or "Delivered," along with any tracking information available.

If you're unable to find the information you need or have any other questions, feel free to reach out to our customer support team at {{Customer Support Phone Number}} or via live chat on our website. We're here to assist you every step of the way!

Customer: My card payment was declined
Answer: I'm sorry to hear that your card was declined and you're experiencing payment problems. I'll do my best to assist you with 

In [ ]:
def chatbot(user_input, top_k=1, threshold=0.55):

    # Semantic retrieval
    results = semantic_search(
        user_input,
        top_k=top_k
    )

    # Best match
    best_result = results.iloc[0]

    score = float(best_result["similarity"])
    intent = best_result["intent"]
    response = best_result["response"]

    # Confidence / unknown-query check
    if score < threshold:

        return {
            "response": (
                "I'm sorry, but I couldn't find enough "
                "information to answer your question. "
                "Could you please rephrase your question?"
            ),
            "intent": "unknown",
            "score": score
        }

    return {
        "response": response,
        "intent": intent,
        "score": score
    }

In [ ]:
test_queries = [
    "Where is my order?",
    "My card payment was declined",
    "I want to return an item",
    "Where is my refund?",
    "I received the wrong product"
]

for query in test_queries:

    result = chatbot(query)

    print("\n" + "=" * 70)
    print("You:", query)
    print("Bot:", result["response"])
    print("Intent:", result["intent"])
    print("Similarity:", round(result["score"], 4))


You: Where is my order?
Bot: I'll do my best! I'd be happy to help you track your order. To check the status of your order, you can follow these steps:

1. Visit our website at {{Company Website URL}}.
2. Log in to your account using your registered email address and password.
3. Navigate to the "{{Order History}}" or "{{My Orders}}" section.
4. Look for the specific order you want to track and click on it.
5. You should be able to see the current status of your order, such as "Processing," "Shipped," or "Delivered," along with any tracking information available.

If you're unable to find the information you need or have any other questions, feel free to reach out to our customer support team at {{Customer Support Phone Number}} or via live chat on our website. We're here to assist you every step of the way!
Intent: track_order
Similarity: 0.9064

You: My card payment was declined
Bot: I'm sorry to hear that your card was declined and you're experiencing payment problems. I'll do my b

In [ ]:
query = "What is the weather today?"

result = chatbot(query)

print("You:", query)
print("Bot:", result["response"])
print("Intent:", result["intent"])
print("Similarity:", round(result["score"], 4))

You: What is the weather today?
Bot: I'm sorry, but I couldn't find enough information to answer your question. Could you please rephrase your question?
Intent: unknown
Similarity: 0.3829


In [ ]:
query = "Can you recommend a laptop for gaming?"

result = chatbot(query)

print("You:", query)
print("Bot:", result["response"])
print("Intent:", result["intent"])
print("Similarity:", round(result["score"], 4))

You: Can you recommend a laptop for gaming?
Bot: I'm sorry, but I couldn't find enough information to answer your question. Could you please rephrase your question?
Intent: unknown
Similarity: 0.2564


In [ ]:
print("*" * 70)
print("E-COMMERCE CUSTOMER SUPPORT CHATBOT")
print("*" * 70)

print("Type 'exit' to end the conversation.\n")

while True:

    user_input = input("You: ")

    if user_input.lower().strip() == "exit":

        print(
            "Bot: Thank you for using the "
            "E-commerce support assistant!"
        )
        break

    result = chatbot(user_input)

    print("\nBot:", result["response"])
    print("=" * 100)
    print()

**********************************************************************
E-COMMERCE CUSTOMER SUPPORT CHATBOT
**********************************************************************
Type 'exit' to end the conversation.

You: where is my order

Bot: I'll do my best! I'd be happy to help you track your order. To check the status of your order, you can follow these steps:

1. Visit our website at {{Company Website URL}}.
2. Log in to your account using your registered email address and password.
3. Navigate to the "{{Order History}}" or "{{My Orders}}" section.
4. Look for the specific order you want to track and click on it.
5. You should be able to see the current status of your order, such as "Processing," "Shipped," or "Delivered," along with any tracking information available.

If you're unable to find the information you need or have any other questions, feel free to reach out to our customer support team at {{Customer Support Phone Number}} or via live chat on our website. We're here t

In [ ]:
import os

# Path to your ONNX model
model_path = onnx_model_path

# Get size
size_bytes = os.path.getsize(model_path)
size_mb = size_bytes / (1024 ** 2)
size_gb = size_bytes / (1024 ** 3)

print(f"ONNX model size: {size_mb:.2f} MB")
print(f"ONNX model size: {size_gb:.4f} GB")

# Check against 5 GB
if size_gb <= 5:
    print("✅ The ONNX model is below the 5 GB limit.")
else:
    print("❌ The ONNX model exceeds the 5 GB limit.")

ONNX model size: 86.26 MB
ONNX model size: 0.0842 GB
✅ The ONNX model is below the 5 GB limit.
